# Text to interpretable features to predictions

In [1]:
import json, sys, time
from pathlib import Path
sys.path.insert(0, "../src")

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from llm_feature_gen import LLMFeatureTransformer
from llm_feature_gen.providers import LocalProvider

posts = pd.read_csv("data/newsgroups_hardware/posts.csv").query("split == 'model'")
X, y = posts["text"].tolist(), posts["label"].tolist()

posts["label"].value_counts()

label
mac    60
pc     60
Name: count, dtype: int64

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
len(X_train), len(X_test)

(90, 30)

In [3]:
provider = LocalProvider()
transformer = LLMFeatureTransformer(provider=provider)

start = time.perf_counter()
F_train = transformer.fit_transform(X_train, y_train)
F_test = transformer.transform(X_test)
elapsed = time.perf_counter() - start

print(f"model: {provider.text_model}")
print(f"{len(X_train)} train + {len(X_test)} test documents -> {F_train.shape[1]} features in {elapsed:.1f}s (discovery and generation)")
F_train.head()

Features saved to outputs/discovered_text_features.json


model: qwen3-vl:32b-instruct
90 train + 30 test documents -> 10 features in 308.0s (discovery and generation)


,hardware_specificity,system_architecture_focus,software_version_reference,upgrade_strategy_discussion,monitor_resolution_concern,power_supply_or_cooling_issue,network_or_communication_protocol,driver_or_firmware_update,user_experience_or_perception,educational_or_academic_context
0,medium,mac_specific,mac_os,new_system_purchase,not_applicable,not_applicable,not_applicable,not_applicable,technical_troubleshooting,not_applicable
1,low,pc_specific,generic,not_applicable,no_resolution,no_physical_issue,pc_network,software_driver,technical_troubleshooting,academic
2,high,general,not_applicable,hardware_upgrade,not_applicable,not_applicable,not_applicable,not_applicable,technical_troubleshooting,not_applicable
3,low,pc_specific,generic,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,technical_troubleshooting,academic
4,medium,pc_specific,pc_os,hardware_upgrade,general_resolution,no_physical_issue,not_applicable,no_update,subjective_experience,not_applicable


In [4]:
list(transformer.get_feature_names_out())

['hardware_specificity',
 'system_architecture_focus',
 'software_version_reference',
 'upgrade_strategy_discussion',
 'monitor_resolution_concern',
 'power_supply_or_cooling_issue',
 'network_or_communication_protocol',
 'driver_or_firmware_update',
 'user_experience_or_perception',
 'educational_or_academic_context']

In [5]:
encoder = OneHotEncoder(handle_unknown="ignore").fit(F_train)
model = LogisticRegression(max_iter=1000).fit(encoder.transform(F_train), y_train)
accuracy = accuracy_score(y_test, model.predict(encoder.transform(F_test)))

print(f"generated features: accuracy {accuracy:.3f} on {len(y_test)} held-out documents")

generated features: accuracy 0.933 on 30 held-out documents


In [6]:
vectorizer = TfidfVectorizer()
baseline = LogisticRegression(max_iter=1000).fit(vectorizer.fit_transform(X_train), y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(vectorizer.transform(X_test)))

print(f"TF-IDF baseline: accuracy {baseline_accuracy:.3f} on {len(y_test)} held-out documents")

TF-IDF baseline: accuracy 0.667 on 30 held-out documents


In [7]:
record = Path("expected/sklearn_notebook")  # record of this run, checked in
record.mkdir(parents=True, exist_ok=True)

(record / "result.json").write_text(json.dumps({
    "model": provider.text_model,
    "n_train": len(X_train),
    "n_test": len(y_test),
    "random_state": 0,
    "features": list(transformer.get_feature_names_out()),
    "seconds_discovery_and_generation": round(elapsed, 1),
    "accuracy_generated_features": round(accuracy, 3),
    "accuracy_tfidf": round(baseline_accuracy, 3),
}, indent=2))
(record / "discovered_features.json").write_text(json.dumps(transformer.discovered_features_, indent=2))

table = pd.concat([
    F_train.assign(split="train", label=y_train, text=X_train),
    F_test.assign(split="test", label=y_test, text=X_test),
])
table.to_csv(record / "feature_table.csv", index=False)
table.shape

(120, 13)